[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kaushikreddym/climdata/blob/main/docs/examples/fair/fair_provenance.ipynb)

In [1]:
# Uncomment to install climdata in Colab or a fresh environment
# !pip install climdata

# FAIR Data Crates with Full Provenance

A NetCDF file on its own does not say how it was made. Which dataset did it come from? Which region and period were cut out? Which index, with which thresholds? Was it regridded, and how? Which version of the software did all this?

`ClimData` records every processing step as it runs. When you save with `to_nc()` or `to_csv()`, the file goes into a folder together with that record. The folder is an [RO-Crate](https://www.researchobject.org/ro-crate/), named after the data it holds:

```text
hyras_hot_spell_frequency_LAT52.3-52.7_LON13.0-13.8_2022-01-01_2023-12-31_crate/
    hyras_hot_spell_frequency_…_2023-12-31.nc    # from to_nc(); CF `history` lists each step
    hyras_hot_spell_frequency_…_2023-12-31.csv   # from to_csv(); same stem, same crate
    meta.json                                    # full record: config, steps, software, checksums
    ro-crate-metadata.json                       # RO-Crate 1.1 JSON-LD, one CreateAction per step
    ro-crate-preview.html                        # human-readable summary; open it in a browser
```

This is the default. There is no separate export step.

This notebook runs a short workflow on real data:

1. **Extract** DWD HYRAS daily maximum temperature around Berlin for 2022–2023.
2. **Reproject** it from the native 1 km ETRS89-LAEA grid to a 0.05° lat/lon grid.
3. **Compute** the annual hot-spell frequency with xclim.
4. **Save** it as NetCDF and CSV. Both files land in one crate.
5. **Inspect** what the crate records, and check it against the data.

HYRAS is free to download from the DWD Open Data server. No account or key is needed. The two yearly files are about 230 MB in total.

## 1. Setup

In [2]:
import html
import json
import os
import warnings
from pathlib import Path

import pandas as pd
import xarray as xr
from IPython.display import HTML, display

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    from climdata import ClimData
    from climdata.utils.provenance import sha256

# Set after the import: importing climdata resets the warning filters.
for category in (FutureWarning, UserWarning, RuntimeWarning):
    warnings.filterwarnings("ignore", category=category)

# Downloads and crates go here. Point CLIMDATA_WORKDIR elsewhere to keep them
# off the current disk.
WORKDIR = Path(os.environ.get("CLIMDATA_WORKDIR", "./data")).resolve()
WORKDIR.mkdir(parents=True, exist_ok=True)

def rel(path):
    """Show paths relative to WORKDIR."""
    return str(Path(path).resolve().relative_to(WORKDIR))

## 2. Configure the run

Everything the workflow does is set through Hydra overrides. Crate settings live under `output.fair`:

| Key | Default |
|-----|---------|
| `output.fair.enabled` | `true`. Set it to `false` to write bare files. |
| `output.fair.title` | `"<DATASET> <variables>"` |
| `output.fair.creator` | none |
| `output.fair.license` | the source dataset's licence, or CC-BY-4.0 if it declares none |
| `output.fair.zip` | `false`. Set it to `true` to also write `<crate>.zip`. |

The configuration is saved in the crate. The full dataset, variable and index catalogues are trimmed down to the entries this run actually used.

In [3]:
overrides = [
    "dataset=hyras",
    "variables=[tasmax]",
    "box.lat_min=52.3", "box.lat_max=52.7",      # Berlin
    "box.lon_min=13.0", "box.lon_max=13.8",
    "time_range.start_date=2022-01-01",
    "time_range.end_date=2023-12-31",
    "index=hot_spell_frequency",                 # xclim: spells of ≥3 days with tasmax > 30 °C
    "target_projection=EPSG:4326",
    "target_resolution=0.05 deg",
    f"data_dir={WORKDIR}",
    f"output.out_dir={WORKDIR / 'crates'}",
    "output.fair.creator=climdata_example",
]
ex = ClimData(overrides=overrides)
# Values with spaces or commas are easier to set on the config directly:
ex.cfg.output.fair.title = "Hot-spell frequency, Berlin, 2022–2023 (HYRAS)"
ex.provenance          # nothing has run yet

[]

## 3. Run the processing steps

Call each step as usual. Every call adds one entry to `ex.provenance`. The entry holds the step's parameters, its UTC start and end times, how long it took, and a summary of what it returned. A step that raises an error is still logged, with status `failed` and the error message.

In [4]:
ds = ex.extract()
ex.provenance[-1]

⬇️  Checking: https://opendata.dwd.de/climate_environment/CDC/grids_germany/daily/hyras_de/air_temperature_max/tasmax_hyras_1_2022_v6-1_de.nc
⬇️  Downloading: https://opendata.dwd.de/climate_environment/CDC/grids_germany/daily/hyras_de/air_temperature_max/tasmax_hyras_1_2022_v6-1_de.nc
✅ Saved: /beegfs/muduchuru/pkgs_fnl/climdata/docs/examples/fair/data/HYRAS/TASMAX/tasmax_hyras_1_2022_v6-1_de.nc
⬇️  Checking: https://opendata.dwd.de/climate_environment/CDC/grids_germany/daily/hyras_de/air_temperature_max/tasmax_hyras_1_2023_v6-1_de.nc
⬇️  Downloading: https://opendata.dwd.de/climate_environment/CDC/grids_germany/daily/hyras_de/air_temperature_max/tasmax_hyras_1_2023_v6-1_de.nc
✅ Saved: /beegfs/muduchuru/pkgs_fnl/climdata/docs/examples/fair/data/HYRAS/TASMAX/tasmax_hyras_1_2023_v6-1_de.nc


{'step': 1,
 'action': 'extract',
 'status': 'completed',
 'started': '2026-10-21T08:59:51Z',
 'ended': '2026-10-21T09:00:13Z',
 'duration_s': 22.582,
 'parameters': {'dataset': 'hyras',
  'provider_class': 'HYRASmirror',
  'variables': ['tasmax'],
  'start_date': '2022-01-01',
  'end_date': '2023-12-31',
  'selection': {'box': {'lat_min': 52.3,
    'lat_max': 52.7,
    'lon_min': 13.0,
    'lon_max': 13.8}},
  'region': None,
  'converted_units': {'tasmax': 'degC'},
  'data_dir': '/beegfs/muduchuru/pkgs_fnl/climdata/docs/examples/fair/data'},
 'output': {'type': 'xarray.Dataset',
  'variables': [{'name': 'tasmax',
    'long_name': 'Daily Maximum Air Temperature',
    'standard_name': 'air_temperature',
    'units': '°C',
    'dims': ['time', 'y', 'x'],
    'shape': [730, 48, 53],
    'dtype': 'float64'}],
  'dims': {'time': 730, 'y': 48, 'x': 53},
  'temporal_coverage': {'start': '2022-01-01', 'end': '2023-12-31'},
  'spatial_coverage': {'lat_min': 52.277489,
   'lat_max': 52.721129,


In [5]:
ex.reproject()
ex.calc_index()

steps = pd.DataFrame(ex.provenance_record())
steps[["step", "action", "status", "started", "duration_s"]]

,step,action,status,started,duration_s
0,1,extract,completed,2026-10-21T08:59:51Z,22.582
1,2,reproject,completed,2026-10-21T09:00:13Z,0.411
2,3,calc_index,completed,2026-10-21T09:00:14Z,2.659


In [6]:
# The parameters each step actually used
for s in ex.provenance_record():
    print(f"{s['step']}. {s['action']}")
    for k, v in s["parameters"].items():
        print(f"     {k:<20} {v}")

1. extract
     dataset              hyras
     provider_class       HYRASmirror
     variables            ['tasmax']
     start_date           2022-01-01
     end_date             2023-12-31
     selection            {'box': {'lat_min': 52.3, 'lat_max': 52.7, 'lon_min': 13.0, 'lon_max': 13.8}}
     region               None
     converted_units      {'tasmax': 'degC'}
     data_dir             /beegfs/muduchuru/pkgs_fnl/climdata/docs/examples/fair/data
2. reproject
     target_projection    EPSG:4326
     target_resolution    0.05 deg
     method               bilinear
     align                True
     bounds               None
     engine               rasterio
     dsinfo_dims          ['x', 'y']
3. calc_index
     index                hot_spell_frequency
     function             xclim.indices.hot_spell_frequency
     input_variables      ['tasmax']
     args                 {'freq': 'YS', 'window': 3}


## 4. Save: the crate is written automatically

`to_nc()` writes the NetCDF into the crate and writes the metadata files next to it. `to_csv()` writes a file with the same name stem, so it goes into the same crate and the metadata is refreshed to include it. Both calls return the path of the data file they wrote. `ex.crate_dir` holds the crate folder.

In [7]:
nc_file = ex.to_nc()
ex.to_dataframe()
csv_file = ex.to_csv()

crate = Path(ex.crate_dir)
print(rel(crate))
for f in sorted(crate.iterdir()):
    print(f"   {f.name:<75} {f.stat().st_size:>9,} bytes")

crates/hyras_hot_spell_frequency_LAT52.3-52.7_LON13.0-13.8_2022-01-01_2023-12-31_crate
   hyras_hot_spell_frequency_LAT52.3-52.7_LON13.0-13.8_2022-01-01_2023-12-31.csv    28,167 bytes
   hyras_hot_spell_frequency_LAT52.3-52.7_LON13.0-13.8_2022-01-01_2023-12-31.nc    26,996 bytes
   meta.json                                                                      11,599 bytes
   ro-crate-metadata.json                                                          9,477 bytes
   ro-crate-preview.html                                                          11,529 bytes


## 5. What the crate records

### `meta.json`: the full record

`meta.json` is the source of truth. The RO-Crate manifest and the HTML preview are both generated from it.

In [8]:
meta = json.loads((crate / "meta.json").read_text())
list(meta)

['schema_version',
 'crate_id',
 'title',
 'description',
 'created',
 'updated',
 'creator',
 'license',
 'source_dataset',
 'source_dataset_info',
 'data',
 'files',
 'processing',
 'config',
 'software']

In [9]:
# The files in the crate, with their checksums
pd.DataFrame(meta["files"])[["path", "role", "size_bytes", "sha256"]]

,path,role,size_bytes,sha256
0,hyras_hot_spell_frequency_LAT52.3-52.7_LON13.0...,data,26996.0,e9a819cc4c6e3c6d81c6b15642573599986ecccc7c51be...
1,hyras_hot_spell_frequency_LAT52.3-52.7_LON13.0...,data,28167.0,abcadd6ea87ae8dc53c8fd4dfb2ab2ab9c6632926ab063...
2,meta.json,metadata,NaN,NaN
3,ro-crate-metadata.json,metadata,NaN,NaN
4,ro-crate-preview.html,metadata,NaN,NaN


In [10]:
# What the data contains
{k: meta["data"][k] for k in ("dims", "temporal_coverage", "spatial_coverage")}

{'dims': {'lon': 19, 'lat': 11, 'time': 2},
 'temporal_coverage': {'start': '2022-01-01', 'end': '2023-01-01'},
 'spatial_coverage': {'lat_min': 52.225,
  'lat_max': 52.725,
  'lon_min': 12.975,
  'lon_max': 13.875}}

In [11]:
# The processing chain, from download to the files on disk
pd.DataFrame(
    [{"step": s["step"], "action": s["action"], "status": s["status"],
      "duration_s": s.get("duration_s"),
      "wrote": ", ".join(s.get("results", []))[:60]}
     for s in meta["processing"]]
)

,step,action,status,duration_s,wrote
0,1,extract,completed,22.582,
1,2,reproject,completed,0.411,
2,3,calc_index,completed,2.659,
3,4,to_nc,completed,0.068,hyras_hot_spell_frequency_LAT52.3-52.7_LON13.0...
4,5,to_dataframe,completed,0.011,
5,6,to_csv,completed,0.008,hyras_hot_spell_frequency_LAT52.3-52.7_LON13.0...


In [12]:
# The extract step, in full
meta["processing"][0]

{'step': 1,
 'action': 'extract',
 'status': 'completed',
 'started': '2026-10-21T08:59:51Z',
 'ended': '2026-10-21T09:00:13Z',
 'duration_s': 22.582,
 'parameters': {'dataset': 'hyras',
  'provider_class': 'HYRASmirror',
  'variables': ['tasmax'],
  'start_date': '2022-01-01',
  'end_date': '2023-12-31',
  'selection': {'box': {'lat_min': 52.3,
    'lat_max': 52.7,
    'lon_min': 13.0,
    'lon_max': 13.8}},
  'region': None,
  'converted_units': {'tasmax': 'degC'},
  'data_dir': '/beegfs/muduchuru/pkgs_fnl/climdata/docs/examples/fair/data'},
 'output': {'type': 'xarray.Dataset',
  'variables': [{'name': 'tasmax',
    'long_name': 'Daily Maximum Air Temperature',
    'standard_name': 'air_temperature',
    'units': '°C',
    'dims': ['time', 'y', 'x'],
    'shape': [730, 48, 53],
    'dtype': 'float64'}],
  'dims': {'time': 730, 'y': 48, 'x': 53},
  'temporal_coverage': {'start': '2022-01-01', 'end': '2023-12-31'},
  'spatial_coverage': {'lat_min': 52.277489,
   'lat_max': 52.721129,


In [13]:
# The configuration this run used, with catalogue entries resolved to what was used
meta["config"]["resolved"]

{'dataset': {'explore': {'full_name': 'Hydrologische Rasterdaten (DWD Gridded)',
   'type': 'Observation (Gridded)',
   'coverage': 'Germany',
   'resolution': '1 km',
   'frequency': 'daily',
   'time_range': '1951-01-01 to present',
   'source': 'DWD open-data server',
   'notes': 'High-resolution gridded climatology over Germany.'},
  'lat_dim': 'y',
  'lon_dim': 'x'},
 'variables': {'tasmax': {'cf_name': 'air_temperature',
   'long_name': 'Daily maximum near-surface air temperature',
   'units': 'degC'}},
 'index': {'hot_spell_frequency': {'function': 'xclim.indices.hot_spell_frequency',
   'variables': ['tasmax'],
   'args': {'freq': 'YS', 'window': 3}}}}

In [14]:
meta["software"]

{'python': '3.10.17',
 'platform': 'Linux-5.14.0-503.35.1.el9_5.x86_64-x86_64-with-glibc2.34',
 'packages': {'climdata': '1.0.0',
  'xarray': '2024.7.0',
  'numpy': '1.26.4',
  'pandas': '2.3.3',
  'xclim': '0.56.0',
  'dask': '2024.8.2',
  'netCDF4': '1.7.2',
  'h5netcdf': '1.8.1',
  'rioxarray': '0.18.2',
  'rasterio': '1.4.3',
  'pyproj': '3.7.1',
  'xesmf': '0.8.8',
  'hydra-core': '1.3.2'}}

### The NetCDF: the history travels with the file

If the NetCDF is copied out of the crate on its own, it still carries a short version of the record. The CF `history` attribute has one line per step, newest first. `climdata_crate_id` links the file back to the crate it came from.

In [15]:
with xr.open_dataset(nc_file) as out:
    print(out.attrs["history"])
    print()
    print("climdata_crate_id:", out.attrs["climdata_crate_id"])
    print("license:          ", out.attrs["license"])

2026-10-21T09:00:16Z climdata 1.0.0 to_nc fair=true
2026-10-21T09:00:16Z climdata 1.0.0 calc_index index=hot_spell_frequency function=xclim.indices.hot_spell_frequency input_variables=["tasmax"] args={"freq":"YS","window":3}
2026-10-21T09:00:14Z climdata 1.0.0 reproject target_projection=EPSG:4326 target_resolution=0.05 deg method=bilinear align=true bounds=null engine=rasterio dsinfo_dims=["x","y"]
2026-10-21T09:00:13Z climdata 1.0.0 extract dataset=hyras provider_class=HYRASmirror variables=["tasmax"] start_date=2022-01-01 end_date=2023-12-31 selection={"box":{"lat_min":52.3,"lat_max":52.7,"lon_min":13.0,"lon_max":13.8}} region=null converted_units={"tasmax":"degC"} data_dir=/beegfs/muduchuru/pkgs_fnl/climdata/docs/examples/fair/data

climdata_crate_id: bf0f11ee-a4dc-41f8-9869-979cb94c23e1
license:           https://creativecommons.org/licenses/by/4.0/


### `ro-crate-metadata.json`: machine-readable

Each step becomes a schema.org `CreateAction`, following the [Process Run Crate](https://www.researchobject.org/workflow-run-crate/profiles/process_run_crate/) profile. Its parameters are listed as `PropertyValue` entries, and the files it wrote are its `result`. Generic RO-Crate tools, repositories such as Zenodo, and workflow registries can read it without knowing anything about climdata.

In [16]:
graph = json.loads((crate / "ro-crate-metadata.json").read_text())["@graph"]
for a in (e for e in graph if e["@type"] == "CreateAction"):
    result = ", ".join(r["@id"].rsplit(".", 1)[-1] for r in a.get("result", []))
    print(f"{a['name']:<16} {a['startTime']} → {a['endTime']}  "
          f"{len(a['additionalProperty'])} parameters" + (f"  → .{result}" if result else ""))

1. extract       2026-10-21T08:59:51Z → 2026-10-21T09:00:13Z  9 parameters
2. reproject     2026-10-21T09:00:13Z → 2026-10-21T09:00:14Z  7 parameters
3. calc_index    2026-10-21T09:00:14Z → 2026-10-21T09:00:16Z  4 parameters
4. to_nc         2026-10-21T09:00:16Z → 2026-10-21T09:00:17Z  1 parameters  → .nc
5. to_dataframe  2026-10-21T09:00:17Z → 2026-10-21T09:00:17Z  0 parameters
6. to_csv        2026-10-21T09:00:17Z → 2026-10-21T09:00:17Z  2 parameters  → .csv


### `ro-crate-preview.html`: human-readable

Open this file in any browser. It shows an overview, the variables, the processing history with every parameter, the files with their checksums, the software versions, and the configuration. It is displayed inline here:

In [17]:
page = (crate / "ro-crate-preview.html").read_text()
display(HTML(f'<iframe srcdoc="{html.escape(page)}" style="width:100%;height:640px;border:1px solid #ccc"></iframe>'))

## 6. Verify the crate

Anyone who receives the crate can check that every data file is the one the record describes:

In [18]:
for f in meta["files"]:
    if f["role"] == "data":
        print(f"{Path(f['path']).suffix:<5} checksum matches: {f['sha256'] == sha256(crate / f['path'])}")

.nc   checksum matches: True
.csv  checksum matches: True


## 7. The same run in one call

`run_workflow` uses the same `to_nc` and `to_csv`, so its output is a crate as well. `result.crate_dir` points to it. Each call to `ClimData(...)` starts a new, empty record.

In [19]:
ex2 = ClimData(overrides=overrides)
result = ex2.run_workflow(actions=["extract", "reproject", "calc_index", "to_nc"])
print(rel(result.crate_dir))
[s["action"] for s in ex2.provenance_record()]

⬇️  Checking: https://opendata.dwd.de/climate_environment/CDC/grids_germany/daily/hyras_de/air_temperature_max/tasmax_hyras_1_2022_v6-1_de.nc
✔️  Exists locally: /beegfs/muduchuru/pkgs_fnl/climdata/docs/examples/fair/data/HYRAS/TASMAX/tasmax_hyras_1_2022_v6-1_de.nc
⬇️  Checking: https://opendata.dwd.de/climate_environment/CDC/grids_germany/daily/hyras_de/air_temperature_max/tasmax_hyras_1_2023_v6-1_de.nc
✔️  Exists locally: /beegfs/muduchuru/pkgs_fnl/climdata/docs/examples/fair/data/HYRAS/TASMAX/tasmax_hyras_1_2023_v6-1_de.nc
crates/hyras_hot_spell_frequency_LAT52.3-52.7_LON13.0-13.8_2022-01-01_2023-12-31_crate


['extract', 'reproject', 'calc_index', 'to_nc']

## 8. Writing a bare file

To write just the file, with no crate, pass `fair=False` for a single call, or set `output.fair.enabled=false` in the configuration:

In [20]:
bare = ex.to_nc(filename=str(WORKDIR / "bare" / "hot_spells.nc"), fair=False)
print(rel(bare))

bare/hot_spells.nc
